# 🚀 在 Google Colab 上使用 vLLM 部署 Hugging Face 开源模型

> **运行前注意**：请在 Colab 顶部菜单依次点击： 或  ->  -> 选择 **T4 GPU**（免费可用），保存后再运行以下单元格。

## 步骤 1：检查 GPU 显存环境

In [ ]:
!nvidia-smi

## 步骤 2：安装 vLLM 与 Hugging Face 工具库

In [ ]:
!pip install -q -U vllm huggingface_hub openai

## 步骤 3：从 Hugging Face 下载轻量开源模型
这里我们选用 **Qwen/Qwen2.5-0.5B-Instruct**（约 940MB，下载极快，同时具备完整的对话指令微调与现代 Transformer 架构）。

In [ ]:
from huggingface_hub import snapshot_download

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
local_dir = "./Qwen2.5-0.5B-Instruct"

print(f"正在从 Hugging Face 下载模型 {model_name}...")
snapshot_download(repo_id=model_name, local_dir=local_dir)
print("
✅ 下载完成！")

## 步骤 4：看看下载下来的模型长什么样？（文件剖析）

In [ ]:
import os, json, struct

print(f"📁 模型目录: {local_dir}
" + "-"*55)
total_size = 0
for f in sorted(os.listdir(local_dir)):
    fp = os.path.join(local_dir, f)
    if os.path.isfile(fp):
        size_mb = os.path.getsize(fp) / (1024 * 1024)
        total_size += size_mb
        print(f"  ├── {f:<26} {size_mb:>8.2f} MB")
print("-"*55)
print(f"总文件大小: {total_size:.2f} MB
")

# 打印 config.json 的关键信息
with open(os.path.join(local_dir, "config.json")) as f:
    cfg = json.load(f)
print("📊 模型核心架构超参数 (config.json):")
print(f"  - 架构类名 (architectures): {cfg.get('architectures')}")
print(f"  - 隐藏层大小 (hidden_size): {cfg.get('hidden_size')}")
print(f"  - 层数 (num_hidden_layers): {cfg.get('num_hidden_layers')}")
print(f"  - 注意力头数 (num_attention_heads): {cfg.get('num_attention_heads')}")
print(f"  - 词表大小 (vocab_size): {cfg.get('vocab_size')}")
print(f"  - 权重精度 (torch_dtype): {cfg.get('torch_dtype')}")

## 步骤 5：使用 vLLM Python SDK 进行极速离线推理
vLLM 的核心优势在于 **PagedAttention** 技术，显存利用率极高，吞吐量相比原生 HuggingFace Transformers 提升数倍至数十倍。

In [ ]:
from vllm import LLM, SamplingParams

# 初始化 vLLM 引擎
llm = LLM(
    model=local_dir,
    dtype="float16",
    gpu_memory_utilization=0.7,
    max_model_len=2048
)

# 设置生成采样参数
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.8,
    max_tokens=256
)

# Qwen 的标准 Chat Prompt 模板
prompts = [
    "<|im_start|>system
你是一个风趣幽默的AI助手。<|im_end|>
<|im_start|>user
用一句话解释量子力学。<|im_end|>
<|im_start|>assistant
",
    "<|im_start|>system
你是一个资深架构师。<|im_end|>
<|im_start|>user
简述 vLLM 的 PagedAttention 是如何解决显存碎片化问题的？<|im_end|>
<|im_start|>assistant
"
]

outputs = llm.generate(prompts, sampling_params)

print("
====== 推理结果 ======
")
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"【Prompt】: {prompt.strip()}")
    print(f"【Output】: {generated_text.strip()}")
    print("-"*50)

## 步骤 6：启动 vLLM 为 OpenAI 兼容的 HTTP API 服务
在实际项目中，我们通常把 vLLM 作为一个服务运行，其他前端、应用或微服务直接像调用 OpenAI API 那样调用它。

In [ ]:
import subprocess, time, requests

# 在后台启动 vLLM OpenAI API Server
serve_cmd = (
    f"python3 -m vllm.entrypoints.openai.api_server "
    f"--model {local_dir} "
    f"--port 8000 "
    f"--gpu-memory-utilization 0.8 "
    f"--max-model-len 2048"
)

print("正在后台启动 vLLM API 服务...")
server_process = subprocess.Popen(serve_cmd, shell=True)

# 健康检查等待启动
for i in range(40):
    try:
        r = requests.get("http://localhost:8000/health")
        if r.status_code == 200:
            print(f"✅ vLLM API 服务已在第 {i*2} 秒就绪！端口: 8000")
            break
    except Exception:
        pass
    time.sleep(2)

## 步骤 7：使用 OpenAI 官方 Python SDK 发起对话请求

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy-token"
)

response = client.chat.completions.create(
    model=local_dir,
    messages=[
        {"role": "system", "content": "你是一个专业的程序员助手。"},
        {"role": "user", "content": "用 Python 写一个反转二叉树的函数。"}
    ],
    temperature=0.7,
    max_tokens=256
)

print(response.choices[0].message.content)